# Notebook 1: 데이터 준비

## 목표
1. jinaai/stanford_slide에서 슬라이드 이미지 10,000장 수집
2. 슬라이드별 썸네일 이미지 생성 (224×224 letterbox)
3. 슬라이드 역할 약한 라벨(weak label) 자동 생성
4. Google Drive에 저장

## 예상 소요 시간
- 스트리밍 수집: 30~60분
- 라벨 생성: 10분

## 0. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = '/content/drive/MyDrive/dadeum_ml'
PPTX_DIR = f'{BASE_DIR}/pptx'
SLIDES_DIR = f'{BASE_DIR}/slides'
LABELS_DIR = f'{BASE_DIR}/labels'
MODELS_DIR = f'{BASE_DIR}/models'

for d in [PPTX_DIR, SLIDES_DIR, LABELS_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

print('디렉토리 구조 생성 완료')
print(f'BASE_DIR: {BASE_DIR}')

## 1. 패키지 설치

In [ ]:
!pip install -q datasets pillow tqdm pandas

# PPTX 파일이 제공되는 경우에만 아래 셀을 실행
# !pip install -q python-pptx
# !apt-get install -q libreoffice

print('설치 완료')

## 2. 데이터셋 구조 탐색

`jinaai/stanford_slide` 필드 이름을 먼저 확인한 뒤 이후 셀 변수를 맞춘다.

In [ ]:
from datasets import load_dataset

print('jinaai/stanford_slide 데이터셋 구조 탐색 중...')
ds_explore = load_dataset('jinaai/stanford_slide', split='train', streaming=True)

first_items = []
for item in ds_explore:
    first_items.append(item)
    if len(first_items) >= 3:
        break

print(f'\n컬럼 목록: {list(first_items[0].keys())}')
print('\n각 컬럼 타입 및 샘플 값:')
for key, val in first_items[0].items():
    print(f'  {key}: {type(val).__name__} = {repr(val)[:120]}')

print('\n--- 덱 구조 파악용: 첫 3개 항목 비교 ---')
CANDIDATE_DECK_FIELDS = ['deck_id', 'presentation_id', 'pptx_id', 'source', 'url', 'id']
CANDIDATE_IDX_FIELDS  = ['slide_index', 'slide_idx', 'page_no', 'page', 'index', 'position']

for i, item in enumerate(first_items):
    print(f'\n항목 {i}:')
    for field in CANDIDATE_DECK_FIELDS + CANDIDATE_IDX_FIELDS:
        if field in item:
            print(f'  {field}: {repr(item[field])[:80]}')

In [ ]:
# ← 위 탐색 결과를 보고 실제 필드 이름으로 수정
DECK_ID_FIELD   = 'deck_id'      # 덱(프레젠테이션) 식별자 필드
SLIDE_IDX_FIELD = 'slide_index'  # 덱 내 슬라이드 순서 필드 (없으면 None)
IMAGE_FIELD     = 'image'        # 이미지 필드 (PIL.Image 또는 bytes)

print(f'DECK_ID_FIELD   = {DECK_ID_FIELD}')
print(f'SLIDE_IDX_FIELD = {SLIDE_IDX_FIELD}')
print(f'IMAGE_FIELD     = {IMAGE_FIELD}')

## 3. 데이터 로드

jinaai/stanford_slide 스트리밍 로딩 (10,000장). 세션 재시작 시 pickle 캐시로 재개.

In [ ]:
from datasets import load_dataset
from pathlib import Path
from PIL import Image
import io, json, pickle
from tqdm import tqdm

TARGET = 10000

# 체크포인트 1: samples 캐시 (Drive에 저장 — 스트리밍 재로딩 방지)
samples_cache = Path(f'{LABELS_DIR}/samples_cache.pkl')

if samples_cache.exists():
    with open(samples_cache, 'rb') as f:
        samples = pickle.load(f)
    print(f'samples 캐시 로드 완료: {len(samples)}개 (스트리밍 재로딩 건너뜀)')
else:
    print(f'jinaai/stanford_slide 스트리밍 로딩 중... (목표: {TARGET}장)')
    ds = load_dataset('jinaai/stanford_slide', split='train', streaming=True)
    samples = []
    for item in tqdm(ds, total=TARGET, desc='샘플 수집'):
        samples.append(item)
        if len(samples) >= TARGET:
            break
    with open(samples_cache, 'wb') as f:
        pickle.dump(samples, f)
    print(f'수집 완료: {len(samples)}개 → {samples_cache} 저장')

print(f'첫 항목 키: {list(samples[0].keys())}')

## 4. 썸네일 추출

이미지 필드에서 슬라이드 PNG를 추출. 비율 유지 후 224×224 letterbox로 저장.

In [ ]:
import pandas as pd
import numpy as np

IMG_SIZE = 224

def _letterbox(img: Image.Image, size: int = 224) -> Image.Image:
    """비율 유지 후 흰색 패딩으로 size×size 캔버스 채움"""
    w, h = img.size
    scale = size / max(w, h)
    new_w, new_h = int(w * scale), int(h * scale)
    img = img.resize((new_w, new_h), Image.LANCZOS)
    canvas = Image.new('RGB', (size, size), (255, 255, 255))
    canvas.paste(img, ((size - new_w) // 2, (size - new_h) // 2))
    return canvas

def extract_image(item: dict) -> Image.Image | None:
    raw = item.get(IMAGE_FIELD)
    if raw is None:
        return None
    if isinstance(raw, bytes):
        return Image.open(io.BytesIO(raw)).convert('RGB')
    if hasattr(raw, 'convert'):
        return raw.convert('RGB')
    return None

def get_deck_id(item: dict, fallback_idx: int) -> str:
    if DECK_ID_FIELD and DECK_ID_FIELD in item:
        return str(item[DECK_ID_FIELD])
    url = item.get('url') or item.get('source') or ''
    if url:
        import hashlib
        return 'deck_' + hashlib.md5(url.encode()).hexdigest()[:8]
    return f'deck_{fallback_idx:05d}'

def get_slide_idx(item: dict) -> int | None:
    if SLIDE_IDX_FIELD and SLIDE_IDX_FIELD in item:
        return int(item[SLIDE_IDX_FIELD])
    return None

all_slide_paths = {}
save_errors = []

for i, item in enumerate(tqdm(samples, desc='이미지 저장')):
    deck_id   = get_deck_id(item, i)
    slide_idx = get_slide_idx(item)
    out_dir   = Path(f'{SLIDES_DIR}/{deck_id}')
    out_dir.mkdir(parents=True, exist_ok=True)
    if slide_idx is None:
        slide_idx = len(list(out_dir.glob('*.png')))
    out_path = str(out_dir / f'slide_{slide_idx:03d}.png')
    if Path(out_path).exists():
        all_slide_paths.setdefault(deck_id, []).append((slide_idx, out_path))
        continue
    img = extract_image(item)
    if img is None:
        save_errors.append({'idx': i, 'deck_id': deck_id, 'reason': 'image_field_missing'})
        continue
    _letterbox(img, IMG_SIZE).save(out_path, 'PNG', optimize=True)
    all_slide_paths.setdefault(deck_id, []).append((slide_idx, out_path))

total_slides = sum(len(v) for v in all_slide_paths.values())
print(f'저장 완료: {len(all_slide_paths)}개 덱, {total_slides}장 슬라이드')
print(f'에러: {len(save_errors)}건')

with open(f'{LABELS_DIR}/save_errors.json', 'w') as f:
    json.dump(save_errors, f, indent=2)

checkpoint_path = Path(f'{LABELS_DIR}/stanford_checkpoint.json')
with open(checkpoint_path, 'w') as f:
    json.dump({'processed': total_slides, 'decks': len(all_slide_paths)}, f)

## 5. 약한 라벨(Weak Label) 자동 생성

python-pptx 없이 위치 정보만으로 역할 라벨 부여.

| 역할 | 규칙 |
|---|---|
| 0: 표지 | 첫 번째 슬라이드 |
| 2: 본문 | 중간 슬라이드 |
| 4: 마무리 | 마지막 슬라이드 |

In [ ]:
ROLE_COVER   = 0
ROLE_SECTION = 1   # 미사용
ROLE_BODY    = 2
ROLE_VISUAL  = 3   # 미사용
ROLE_CLOSING = 4
ROLE_NAMES   = ['표지', '섹션헤더', '본문', '도표/시각자료', '마무리']

def assign_weak_label_image(slide_idx: int, total_slides: int) -> int:
    if slide_idx == 0:
        return ROLE_COVER
    if slide_idx == total_slides - 1:
        return ROLE_CLOSING
    return ROLE_BODY

records = []
for deck_id, slides in tqdm(all_slide_paths.items(), desc='약한 라벨 생성'):
    slides_sorted = sorted(slides, key=lambda x: x[0])
    n = len(slides_sorted)
    for rank, (slide_idx, img_path) in enumerate(slides_sorted):
        label = assign_weak_label_image(rank, n)
        records.append({
            'deck_id':        deck_id,
            'slide_idx':      rank,
            'total_slides':   n,
            'position_ratio': round(rank / max(n - 1, 1), 4),
            'word_count':     0,
            'visual_ratio':   0.0,
            'has_table':      False,
            'has_chart':      False,
            'weak_label':     label,
            'role_name':      ROLE_NAMES[label],
            'image_path':     img_path,
        })

df = pd.DataFrame(records)
print(f'총 슬라이드: {len(df)}장')
print(df['role_name'].value_counts())
print('\n⚠ 클래스 1(섹션헤더), 3(도표)은 미사용 — Notebook 02 class_weights 주의')

## 6. 클래스 가중치 저장

In [ ]:
print('\n=== 클래스 분포 체크 ===')
for role, ratio in df['role_name'].value_counts(normalize=True).items():
    flag = ' ⚠ 소수 클래스' if ratio < 0.10 else ''
    print(f'  {role}: {ratio:.1%}{flag}')

class_counts = df['weak_label'].value_counts().sort_index()

# 부재 클래스(1, 3)는 weight=0.0 — CrossEntropyLoss가 자동 무시
weights = {str(i): float(1.0 / class_counts[i]) if i in class_counts.index else 0.0
           for i in range(5)}
with open(f'{LABELS_DIR}/class_weights.json', 'w') as f:
    json.dump(weights, f, indent=2)

df.to_csv(f'{LABELS_DIR}/weak_labels.csv', index=False)
print(f'저장 완료: {LABELS_DIR}/weak_labels.csv')
print(f'클래스 가중치: {weights}')

In [ ]:
# 샘플 시각화
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for role_id, ax in enumerate(axes):
    subset = df[df['weak_label'] == role_id]
    if len(subset) == 0:
        ax.set_title(f'{ROLE_NAMES[role_id]}\n(샘플 없음)')
        ax.axis('off')
        continue
    sample = subset.sample(1).iloc[0]
    img = mpimg.imread(sample['image_path'])
    ax.imshow(img)
    ax.set_title(f'{ROLE_NAMES[role_id]}\n({len(subset)}장)', fontsize=10)
    ax.axis('off')

plt.suptitle('역할별 슬라이드 샘플', fontsize=14)
plt.tight_layout()
plt.savefig(f'{LABELS_DIR}/role_samples.png', dpi=100)
plt.show()
print('시각화 저장 완료')

## 7. PPTX 경로 체크

stanford_slide는 이미지 전용 — PPTX 없음. Step 1(data-prep) skip.

In [ ]:
# PPTX 데이터 여부 확인
has_pptx = len(list(Path(PPTX_DIR).glob('*.pptx'))) > 0
print(f'PPTX 데이터 존재: {has_pptx}')
if not has_pptx:
    print('⚠ stanford_slide는 이미지 전용 데이터셋입니다.')
    print('  Step 1(data-prep)은 PPTX 경로에만 해당 — 자동 skip됩니다.')

## 8. HMM 학습용 시퀀스 데이터 생성

덱별 역할 시퀀스를 추출해서 Notebook 3 HMM 학습에 사용할 데이터 저장.

In [ ]:
sequences = []

for deck_id, group in df.groupby('deck_id'):
    group = group.sort_values('slide_idx')
    seq = group['weak_label'].tolist()
    if len(seq) >= 2:  # 최소 2장 이상인 덱만 (1장 덱은 HMM 학습에 불필요)
        sequences.append({'deck_id': deck_id, 'sequence': seq, 'length': len(seq)})

seq_df = pd.DataFrame(sequences)
seq_df.to_csv(f'{LABELS_DIR}/sequences.csv', index=False)

print(f'시퀀스 수: {len(seq_df)}')
print(f'평균 덱 길이: {seq_df["length"].mean():.1f}장')
print(f'최대 덱 길이: {seq_df["length"].max()}장')

from collections import Counter
short_seqs = seq_df[seq_df['length'] <= 10]['sequence'].tolist()
pattern_counts = Counter([tuple(s) for s in short_seqs])
print('\n가장 흔한 시퀀스 패턴 (10장 이하):')
for pattern, count in pattern_counts.most_common(5):
    readable = ' → '.join([ROLE_NAMES[r] for r in pattern])
    print(f'  {readable}  ({count}개)')

In [ ]:
label_path = f'{LABELS_DIR}/weak_labels.csv'
print('=== Notebook 1 완료 ===')
print(f'슬라이드 이미지: {len(df)}장 ({len(all_slide_paths)}개 덱)')
print(f'약한 라벨 CSV: {label_path}')
print(f'클래스 가중치: {LABELS_DIR}/class_weights.json')
print(f'시퀀스 CSV: {LABELS_DIR}/sequences.csv')
print('\nNotebook 2 (CNN 역할 분류기)로 이동하세요.')